# *Libraries*

In [2]:
from typing import TypedDict, Optional

import os

import fitz

# *Agent State*

In [3]:
class AgentState(TypedDict):
    # File
    file_path : Optional[str]
    file_name : str
    file_type : str

    # Validation
    validation_status : str
    validation_reason : str

# *Upload API Node*

In [19]:
MAX_FILE_SIZE = 10

VALID_EXTENSION = [".pdf", ".txt", ".docx"]



In [20]:
def upload_api(state : AgentState, file_path : str) -> AgentState:
    """
    Validates uploaded document before entering RAG pipeline.

    Checks:
    - file existence
    - supported extension
    - file size limit

    Updates workflow state with validation results.
    """

    # File Existence Check
    if not os.path.exists(file_path):

        state["validation_status"] = "failed"
        state["validation_reason"] = "file_not_found"

        return state
    
    # Retreive File Name and Extension
    file_name = os.path.basename(file_path)

    _, extension = os.path.splitext(file_name)

    # File Extensions Check
    if extension.lower() not in VALID_EXTENSION:

        state["validation_status"] = "failed"
        state["validation_reason"] = "unsupported_file_type"

        return state
    
    # File Size Check
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)

    if file_size_mb > MAX_FILE_SIZE:

        state["validation_status"] = "failed"
        state["validation_reason"] = "file_too_large"

        return state

    # Success 
    state["file_path"] = file_path
    state["file_name"] = file_name
    state["file_type"] = extension.lower()

    state["validation_status"] = "success"
    state["validation_reason"] = ""

    return state

# *Document Validation Node*

In [5]:
def document_validation_node(state : AgentState) -> AgentState:
    """
    Validates whether uploaded document
    is safe and usable for the RAG pipeline.

    Checks:
    - corrupted PDF
    - encrypted PDF
    - extractable text
    - empty document
    - scanned/image-only PDF
    """

    # Reterive file info from state
    file_path = state["file_path"]
    file_type = state["file_type"]

    # Validates .txt and .docx files
    if file_type  in [".txt", ".docx"]:

        state["validation_status"] = "success"
        
        return state
    
    # Validates PDF's
    # Corrupted pdf check
    try:
        # Open pdf
        doc = fitz.open(file_path)

    except Exception:
        state["validation_status"] = "failed"
        state["validation_reason"] = "corrupted_pdf"

        return state
    
    # Encrypted pdf check
    if doc.is_encrypted():
        state["validation_status"] = "failed"
        state["validation_reason"] = "encrypted_pdf"

        return state
    
    # Extract text
    extracted_text = ""

    for page in doc:
        extracted_text += page.get_text()

    # Empty pdf check
    if len(extracted_text.strip()) == 0:
        state["validation_status"] = "failed"
        state["validation_reason"] = "empty_pdf"

        return state  

    # Scanned/Image only pdf check
    if len(extracted_text.strip()) < 100:
        state["validation_status"] = "failed"
        state["validation_reason"] = "scanned_or_image_only_pdf"

        return state  
    
    # Success
    state["validation_status"] = "success"
    state["validation_reason"] = ""

    doc.close()

    return state    
    